# Supplementary Figures S2 and S16: snapshots of the citation preference network

The network pictures of Supplementary Fig. S2 (2000 and 2010) and Supplementary Fig. S16 (2000, 2005, 2010 and 2017) were drawn in Gephi with the node placement of Fig. 2. This notebook exports the cumulative positive citation preference networks of these years as gexf files (in `figures/networks`), from 5_GlobalCitationNetwork_inclusionfragmentation.

It reads its input from the data folder (`path2data`). The figures are saved in the `figures` folder.

In [1]:
import os
import json
import gzip
import itertools
from itertools import chain
from collections import defaultdict
import pandas as pd
import numpy as np
import networkx as nx

import scipy.sparse as spsparse
import scipy.stats as spstats

from statsmodels.stats.multitest import multipletests as holm
import statsmodels.api as sm

import matplotlib.pylab as plt
import matplotlib.gridspec as gridspec
import matplotlib.transforms as mtransforms
import seaborn as sns

# set this path to where the processed data (Source Data) is stored:
path2sourcedata = "Source Data"
# when the notebook is run from the code folder, the Source Data folder is one level up
if not os.path.exists(path2sourcedata):
    path2sourcedata = os.path.join("..", "Source Data")

# the processed input data are in the data folder, next to the Source Data folder
path2data = os.path.join(os.path.dirname(os.path.abspath(path2sourcedata)), 'data')

# the regenerated figures are saved in the figures folder
path2figures = os.path.join(os.path.dirname(os.path.abspath(path2sourcedata)), 'figures')
os.makedirs(path2figures, exist_ok=True)

# the networks are exported for Gephi
path2networks = os.path.join(path2figures, 'networks')
os.makedirs(path2networks, exist_ok=True)

In [2]:
# Country counts per year for four conditions

def get_yearly_net(publim, syear=1990, eyear=2017):
    country_auc_df = pd.read_csv(os.path.join(path2data, 'oa_countrycites_nosameorg_auc.csv.gz'))
    # the years are stored as decimals in the csv file
    country_auc_df['CitedYear'] = country_auc_df['CitedYear'].astype(int)
    country_auc_df['Zscore'] = (country_auc_df['AUC'] - 0.5) / np.sqrt(country_auc_df['Cov'])
    country_auc_df = country_auc_df[country_auc_df['N'] >= publim].reset_index(drop=True)
    # one-sided test in the direction of the observed preference: H1 is AUC > 0.5 (over-citation)
    # when Z > 0, and AUC < 0.5 (under-citation) when Z < 0
    country_auc_df['pvalue'] = spstats.norm.sf(np.abs(country_auc_df['Zscore'].values))

    def reject_holm(pvalues):
        reject, _, _, _ = holm(pvalues, alpha=0.01, method='hs', is_sorted=False, returnsorted=False)
        return reject

    country_auc_df['Significant'] = country_auc_df.groupby('CitedYear')['pvalue'].transform(reject_holm)

    # eSign: 1 = significant positive, -1 = significant negative, 0 = not significant
    country_auc_df['eSign'] = 0
    country_auc_df.loc[(country_auc_df['Significant'] == True) & (country_auc_df['Zscore'] > 0), 'eSign'] = 1
    country_auc_df.loc[(country_auc_df['Significant'] == True) & (country_auc_df['Zscore'] < 0), 'eSign'] = -1

    net = country_auc_df[country_auc_df['CitingCountry'] != country_auc_df['CitedCountry']]
    net = net[(net['CitedYear'] >= syear) & (net['CitedYear'] <= eyear)].copy()

    return net

def count_by_year(net, esign_filter=None, esign_min=None):
    
    """Cumulative network: for each year Y, 
    keep most recent eSign per (Citing, Cited) pair,
    then count unique countries."""
    
    years = sorted(net['CitedYear'].unique())
    results = []
    for y in years:
        cumnet = (net[net['CitedYear'] <= y]
                  .sort_values('CitedYear')
                  .drop_duplicates(subset=['CitingCountry', 'CitedCountry'], keep='last'))
        if esign_filter is not None:
            cumnet = cumnet[cumnet['eSign'] == esign_filter]
        elif esign_min is not None:
            cumnet = cumnet[cumnet['eSign'] >= esign_min]
        countries = set(cumnet['CitingCountry']).union(set(cumnet['CitedCountry']))
        results.append({'Year': int(y), 'n_countries': len(countries)})
    return pd.DataFrame(results)

yearly50 = get_yearly_net(50)
yearly2  = get_yearly_net(2)

c1 = count_by_year(yearly50, esign_filter=1).rename(columns={'n_countries': 'publim50_eSign_pos'})
c2 = count_by_year(yearly50, esign_min=0).rename(columns={'n_countries': 'publim50_eSign_nonneg'})
c3 = count_by_year(yearly50).rename(columns={'n_countries': 'publim50_all'})
c4 = count_by_year(yearly2).rename(columns={'n_countries': 'publim2_all'})

yearly_counts = c1.merge(c2, on='Year').merge(c3, on='Year').merge(c4, on='Year')
yearly_counts

,Year,publim50_eSign_pos,publim50_eSign_nonneg,publim50_all,publim2_all
0,1990,8,97,97,182
1,1991,9,100,100,190
2,1992,9,101,101,197
3,1993,12,104,104,202
4,1994,12,108,108,206
5,1995,17,113,113,209
6,1996,16,119,119,209
7,1997,18,124,124,210
8,1998,17,133,133,210
9,1999,18,137,138,213


In [3]:
def cumnet_by_year(net, esign_filter=None):
    """Cumulative network: for each year Y,
    keep most recent eSign per pair,
    then count unique countries."""
    years = sorted(net['CitedYear'].unique())
    #years = [i for i in years if i >= base_year]
    results = []
    for year in years:
        
        cumnet = (net[net['CitedYear'] <= year]
                  .sort_values('CitedYear')
                  .drop_duplicates(subset=['CitingCountry', 'CitedCountry'], keep='last'))
        if esign_filter is not None:
            cumnet = cumnet[cumnet['eSign'] == esign_filter]
            cumnet['year']=year
            
        results.append(cumnet)
    return pd.concat(results)

cumnet_df = cumnet_by_year(yearly50,1)

In [4]:
# Years to export
years_to_export = [2000,2005, 2010, 2017]

for year in years_to_export:
    # Select the cumulative network for this year
    df_year = cumnet_df[cumnet_df["year"] == year]

    # Create graph
    G = nx.from_pandas_edgelist(
        df_year,
        source="CitingCountry",
        target="CitedCountry",
        edge_attr=True,          # keep all edge attributes
        create_using=nx.DiGraph()  # or nx.Graph() if undirected
    )

    # Export
    nx.write_gexf(G, os.path.join(path2networks, f"pos_network_{year}.gexf"))

    print(f"{year}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

2000: 28 nodes, 34 edges
2005: 34 nodes, 82 edges
2010: 54 nodes, 182 edges
2017: 78 nodes, 434 edges


## Source Data for Supplementary Figs. S2 and S16

The positive edges of the cumulative networks drawn in Gephi, written to `Source Data/SourceData_FigS2.xlsx` (2000 and 2010) and `SourceData_FigS16.xlsx` (2000, 2005, 2010 and 2017).

In [5]:
from sourcedatautils import write_source_data, country2commlabel

def network_edges(years):
    df = cumnet_df[cumnet_df['year'].isin(years)][['year', 'CitingCountry', 'CitedCountry', 'CitedYear', 'AUC', 'Zscore']]
    df = df.rename(columns={'year':'Network year', 'CitingCountry':'Citing country', 'CitedCountry':'Cited country',
                            'CitedYear':'Year of the most recent significant preference', 'Zscore':'Z-score'})
    df['Citing community'] = df['Citing country'].map(country2commlabel)
    df['Cited community'] = df['Cited country'].map(country2commlabel)
    return df.reset_index(drop=True)

write_source_data('SourceData_FigS2.xlsx', 'Supplementary Figure 2: Snapshots of the international citation preference network.',
                  ('Positive edges of the cumulative citation preference network up to 2000 (A) and 2010 (B), with the '
                   'communities (1-5, as in Fig. 2) of the citing and cited countries.'),
                  [('S2', 'Positive edges of the cumulative network in each year.', network_edges([2000, 2010]))])

write_source_data('SourceData_FigS16.xlsx', 'Supplementary Figure 16: Transition from a centralized to a fragmented scientific recognition network.',
                  ('Positive edges of the cumulative citation preference network up to 2000 (A), 2005 (B), 2010 (C) and '
                   '2017 (D); node size in the figure is the degree.'),
                  [('S16', 'Positive edges of the cumulative network in each year.', network_edges([2000, 2005, 2010, 2017]))])

wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/SourceData_FigS2.xlsx
wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/README.txt
wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/SourceData_FigS16.xlsx
wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/README.txt
